In [ ]:
%pip install --upgrade pip

# Uninstall conflicting packages
%pip uninstall -y langchain_classic langchain-core langchain-openai langchain-community langchain langchain-chroma chromadb beautifulsoup4 python-dotenv PyPDF2 rank_bm25 weaviate-client ragas wikipedia langchain-weaviate langchain-together gradio 6.0.2

# Step 2.3: Install required packages
%pip install neo4j==6.0.3
%pip install rdflib==7.5.0
%pip install pandas==2.3.3
%pip install sentence-transformers==5.1.2
%pip install faiss-cpu==1.13.1
%pip install python-dotenv==1.2.1
%pip install langchain==1.1.0
%pip install langchain-openai==1.1.0
%pip install langchain-community==0.4.1

In [ ]:
# Step 3.2: Set up imports
import os
import rdflib
from rdflib.namespace import RDF, RDFS, OWL
import pandas as pd
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Any
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import textwrap

# Load env vars from the file used in previous chapters
_ = load_dotenv(dotenv_path='env.txt')
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

# Neo4j setup
NEO4J_URI = os.getenv('NEO4J_URI', 'neo4j://127.0.0.1:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASS = os.getenv('NEO4J_PASS', 'password')

# LLM setup
CHAT_MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=CHAT_MODEL, temperature=0.2)
os.environ["LANGCHAIN_TRACING_V2"] = "false"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

def reset_neo4j_database():
    """Remove all nodes and relationships from Neo4j"""
    with driver.session() as session:
        result = session.run("MATCH (n) DETACH DELETE n")
        summary = result.consume()
        print(f"Neo4j database reset - deleted {summary.counters.nodes_deleted} nodes and {summary.counters.relationships_deleted} relationships")

# --- Test connection and optionally reset ---
try:
    with driver.session() as session:
        session.run("RETURN 1")
    print(f"Connected to Neo4j at {NEO4J_URI} as user {NEO4J_USER}")
    
    # Uncomment the line below to reset your database
    # WARNING: This will delete ALL data in your Neo4j instance
    # reset_neo4j_database()
    
except Exception as e:
    print(f"WARNING: Could not connect to Neo4j at {NEO4J_URI}")
    print(f"Make sure Neo4j Desktop is running and the database is started.")
    print(f"Error: {e}")

In [ ]:
# Step 3.3: Load the ontology
g = rdflib.Graph()
g.parse('FinancialOntology.ttl', format='turtle')

# Helper to get first value of a given property
def get_first(subject, prop):
    for val in g.objects(subject, prop):
        return str(val)
    return None

# --- Collect nodes ---
nodes = []
for s in g.subjects(RDF.type, OWL.Class):
    nodes.append({
        'id': str(s),
        'label': get_first(s, RDFS.label) or s.split('#')[-1],
        'comment': get_first(s, RDFS.comment),
        'type': 'Class'
    })
for s in g.subjects(RDF.type, OWL.NamedIndividual):
    nodes.append({
        'id': str(s),
        'label': get_first(s, RDFS.label) or s.split('#')[-1],
        'comment': get_first(s, RDFS.comment),
        'type': 'Individual'
    })
nodes_df = pd.DataFrame(nodes)
nodes_df.to_csv('ontology_nodes.csv', index=False)

